<a id="top"></a>

# Demo: trace the round-trip, message by message

```yaml
title: "Demo: trace the round-trip, message by message"
keywords: round-trip, four messages, conversation history, call id, stateless, token cost, trace, teacher demo
estimated duration: 10
```

> **Lesson:** L07. Teacher demo — Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). **No API key needed** — this demo dissects a *crafted transcript* of the exact exchange [Demo 2](L0404_lecture.ipynb) produces, so the message structure is identical every run. (In class you can instead replay the real `messages` list Demo 2 built.)
>
> The point to land: a single tool-using exchange is **at minimum four messages**, and the call id is the thread tying request to result. Every tool call grows the history — that is the protocol, not a side effect.

## Contents

- [1. Setup — a captured four-message transcript](#1-setup--a-captured-four-message-transcript)
- [2. Walk the four messages in order](#2-walk-the-four-messages-in-order)
- [3. The id ties the result to the request](#3-the-id-ties-the-result-to-the-request)
- [4. Count the growth](#4-count-the-growth)
- [5. Takeaways](#5-takeaways)

## 1. Setup — a captured four-message transcript

The same shape Demo 2 produced, written out as plain dicts so we can walk it slowly with no live call. Block types are shown as the SDK reports them (`text`, `tool_use`, `tool_result`).

In [ ]:
# A captured transcript of one successful round-trip (Demo 2's exchange).
# Roles and block types match what the Anthropic SDK produced.
CALL_ID = "toolu_01ABC"  # the unique id the model assigned to its tool call

transcript: list[dict[str, object]] = [
    {
        "role": "user",
        "content": "What is 18,374 multiplied by 92,431?",
    },
    {
        "role": "assistant",
        "content": [
            {
                "type": "tool_use",
                "id": CALL_ID,
                "name": "calculator",
                "input": {"expression": "18374 * 92431"},
            },
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "tool_result", "tool_use_id": CALL_ID, "content": "1698367394"},
        ],
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": "18,374 x 92,431 = 1,698,367,394."},
        ],
    },
]
print(f"{len(transcript)} messages captured")

[↑ Back to top](#top)

## 2. Walk the four messages in order

Print each message with its role, the block types it carries, and **who produced it**. The user 'asked once' — yet the history is four messages long.

In [ ]:
def block_types(content: object) -> str:
    """Summarize a message's content as a list of block types (or 'text')."""
    if isinstance(content, str):
        return "text"
    assert isinstance(content, list)
    return ", ".join(str(b["type"]) for b in content)


producers = ["the human", "the model", "the APPLICATION (wearing the user role)", "the model"]
for i, (msg, who) in enumerate(zip(transcript, producers, strict=True), start=1):
    print(f"Message {i} | role={msg['role']:9} | blocks: {block_types(msg['content'])}")
    print(f"           produced by {who}")

[↑ Back to top](#top)

## 3. The id ties the result to the request

The `tool_result` in Message 3 names the **same** id as the `tool_use` in Message 2. That id is how the application says 'this result answers *that* request' — essential once more than one call is in flight ([L10](L0402_lecture.md)).

In [ ]:
use_block = transcript[1]["content"][0]  # the tool_use in message 2
result_block = transcript[2]["content"][0]  # the tool_result in message 3
print("tool_use id     (msg 2):", use_block["id"])
print("tool_result id  (msg 3):", result_block["tool_use_id"])
assert use_block["id"] == result_block["tool_use_id"]
print("ids match -> this result answers that exact request")

[↑ Back to top](#top)

## 4. Count the growth

The user asked **once**, but the conversation grew by **four** messages. Every future tool call repeats this growth — and the tool *definition* is re-sent on every request because the model is **stateless**.

In [ ]:
print("user questions asked     :", 1)
print("messages now in history  :", len(transcript))
print()
# Tools cost tokens TWICE OVER: the definition rides in every request, and the
# result lives in the history for every later turn. A 500-token definition across
# a 10-turn chat is ~5,000 input tokens before the tool is even called.
DEFINITION_TOKENS = 500
TURNS = 10
print(f"~{DEFINITION_TOKENS * TURNS} input tokens spent on the definition alone over {TURNS} turns")

[↑ Back to top](#top)

## 5. Takeaways

- A successful round-trip is **four messages**: `user -> assistant(tool_use) -> user(tool_result) -> assistant(final)`. Four is the *minimum*, not a fixed number.
- The conversation alternates **assistant <-> user** even though one 'user' is the application. Role labels mark **protocol position**, not who is human.
- The model is **stateless across calls** — Message 4 required re-sending Messages 1-3 *and* the tool definition. The model did not 'remember' the tool.
- Tools cost tokens **twice over** — definition re-sent every request, result kept in history. Forward-link to [L12](L0402_lecture.md) (model power) and L16 (context management).

[↑ Back to top](#top)